# 第11章 树模型与集成学习 —— 配套代码

对应正文 `book/part3/11-ensemble-trees.md`。

> **运行前准备**：请先在终端执行 `uv run python scripts/make_sample_data.py` 生成内置示例数据。

## 演示内容

1. 环境初始化与数据加载
2. 特征工程：构造技术指标特征
3. 单棵决策树：深度与过拟合演示
4. 随机森林：OOB 误差与特征重要性
5. XGBoost：时序交叉验证调参 + 早停
6. XGBoost vs 逻辑回归：AUC 对比 + 混淆矩阵
7. 特征重要性条形图（Gain / Weight / Cover）
8. 置换重要性（SHAP 替代方案）
9. 偏差-方差可视化（Bootstrap）
10. 超参敏感性热图
11. 综合汇总
12. 习题参考解答


In [ ]:
# Cell 1：环境初始化与数据加载
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (
    roc_auc_score, ConfusionMatrixDisplay, classification_report, roc_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier

from fds import load_sample_prices, daily_returns, set_chinese_font

set_chinese_font()

SEED = 42
np.random.seed(SEED)

prices = load_sample_prices()
rets   = daily_returns(prices)

print(f'价格数据维度：{prices.shape}')
print(f'收益率维度：{rets.shape}')
print(f'日期范围：{rets.index[0].date()} → {rets.index[-1].date()}')
print(f'股票列：{list(rets.columns)}')
rets.tail(3)


## 11.1 特征工程：构造技术指标特征

以 **TECH** 单股为实战对象，构造以下预测特征：

| 特征 | 含义 |
|------|------|
| `ret_1` | 昨日收益率（短线反转信号） |
| `mom_5` | 5 日动量（短期惯性） |
| `mom_20` | 20 日动量（月度动量） |
| `vol_5` | 5 日滚动波动率 |
| `vol_20` | 20 日滚动波动率 |
| `ma_ratio` | 收盘价 / 20 日均价偏离（偏离程度） |
| `vol_ratio` | 5 日波动率 / 20 日波动率（波动变化趋势） |

预测目标 $y = \mathbf{1}[\text{次日收益率} > 0]$（二分类）。


In [ ]:
# Cell 2：特征工程
def build_features(ret_series, price_series):
    """构造技术特征并拼接目标变量"""
    df = pd.DataFrame(index=ret_series.index)
    df['ret_1']    = ret_series.shift(1)
    df['mom_5']    = ret_series.shift(1).rolling(5).sum()
    df['mom_20']   = ret_series.shift(1).rolling(20).sum()
    df['vol_5']    = ret_series.shift(1).rolling(5).std()
    df['vol_20']   = ret_series.shift(1).rolling(20).std()
    df['ma_ratio'] = (price_series / price_series.rolling(20).mean() - 1).shift(1)
    df['vol_ratio'] = df['vol_5'] / (df['vol_20'] + 1e-9)
    df['target']   = (ret_series > 0).astype(int)
    return df.dropna()

STOCK = 'TECH'
FEATURE_COLS = ['ret_1', 'mom_5', 'mom_20', 'vol_5', 'vol_20', 'ma_ratio', 'vol_ratio']

feat_df = build_features(rets[STOCK], prices[STOCK])
X_all   = feat_df[FEATURE_COLS].values
y_all   = feat_df['target'].values

n_total = len(feat_df)
n_train = int(n_total * 0.70)
X_train, X_test = X_all[:n_train], X_all[n_train:]
y_train, y_test = y_all[:n_train], y_all[n_train:]
dates_train = feat_df.index[:n_train]
dates_test  = feat_df.index[n_train:]

print(f'总样本数：{n_total}')
print(f'训练集：{n_train} 条 ({dates_train[0].date()} → {dates_train[-1].date()})')
print(f'测试集：{n_total - n_train} 条 ({dates_test[0].date()} → {dates_test[-1].date()})')
print(f'涨跌比例：涨={y_all.mean():.2%}，跌={1-y_all.mean():.2%}')
feat_df[FEATURE_COLS + ['target']].head()


## 11.2 单棵决策树：深度与过拟合演示

通过改变 `max_depth`，观察训练/测试 AUC 的剪刀差现象。

> **警告**：金融收益序列信噪比极低，单棵深树几乎必然过拟合。训练 AUC 可达 0.9+，测试 AUC 却可能低于随机猜测。


In [ ]:
# Cell 3：决策树深度 vs 过拟合
depths = [1, 2, 3, 4, 5, 6, 8, 10, 15]
train_aucs, test_aucs = [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=SEED)
    dt.fit(X_train, y_train)
    train_aucs.append(roc_auc_score(y_train, dt.predict_proba(X_train)[:, 1]))
    test_aucs.append(roc_auc_score(y_test,   dt.predict_proba(X_test)[:, 1]))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(depths, train_aucs, 'o-', color='steelblue', label='训练 AUC', lw=2)
ax.plot(depths, test_aucs,  's--', color='tomato',    label='测试 AUC',  lw=2)
ax.axhline(0.5, color='gray', linestyle=':', lw=1, label='随机猜测（0.5）')
ax.set_xlabel('决策树最大深度 (max_depth)')
ax.set_ylabel('AUC')
ax.set_title(f'{STOCK} 涨跌预测：决策树深度 vs AUC（训练/测试）')
ax.legend()
ax.set_ylim(0.45, 1.02)
plt.tight_layout()
plt.show()

df_depth = pd.DataFrame({'深度': depths, '训练AUC': np.round(train_aucs, 4),
                          '测试AUC': np.round(test_aucs, 4)})
df_depth['差距'] = (df_depth['训练AUC'] - df_depth['测试AUC']).round(4)
print(df_depth.to_string(index=False))


## 11.3 随机森林：OOB 误差与特征重要性

随机森林通过 `oob_score=True` 启用袋外误差评估，无需额外验证集。

**特征重要性**（不纯度下降量）反映各特征对分裂的平均贡献。


In [ ]:
# Cell 4：随机森林 —— OOB 误差 & 特征重要性
rf = RandomForestClassifier(
    n_estimators=300, max_depth=5, min_samples_leaf=20,
    max_features='sqrt', oob_score=True,
    random_state=SEED, n_jobs=1
)
rf.fit(X_train, y_train)

rf_train_auc = roc_auc_score(y_train, rf.predict_proba(X_train)[:, 1])
rf_test_auc  = roc_auc_score(y_test,  rf.predict_proba(X_test)[:, 1])

print(f'随机森林（300棵，max_depth=5）：')
print(f'  训练 AUC：{rf_train_auc:.4f}')
print(f'  测试 AUC ：{rf_test_auc:.4f}')
print(f'  OOB 准确率：{rf.oob_score_:.4f}')

feat_imp = pd.Series(rf.feature_importances_, index=FEATURE_COLS)
feat_imp = feat_imp.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(feat_imp)))
bars = ax.barh(feat_imp.index[::-1], feat_imp.values[::-1], color=colors[::-1])
ax.set_xlabel('特征重要性（不纯度下降量）')
ax.set_title('随机森林特征重要性（Impurity Importance）')
for bar, val in zip(bars, feat_imp.values[::-1]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()


## 11.4 XGBoost：时序交叉验证调参

使用 `TimeSeriesSplit` 进行时序 5 折交叉验证，在训练集内搜索最优超参数。

**关键超参**：`max_depth`（3～5）、`learning_rate`（0.01～0.1）。

> 时序 CV 的每一折均保证验证集在训练集之后，避免前视偏差。


In [ ]:
# Cell 5：时序交叉验证 + 网格搜索调参
tscv = TimeSeriesSplit(n_splits=5)

param_grid = {
    'max_depth':     [3, 4, 5],
    'learning_rate': [0.01, 0.05, 0.10],
}

base_xgb = XGBClassifier(
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    eval_metric='logloss',
    random_state=SEED,
    verbosity=0,
)

grid_search = GridSearchCV(
    base_xgb, param_grid,
    cv=tscv, scoring='roc_auc',
    n_jobs=1, verbose=0, return_train_score=True
)
grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
print(f'最优超参：{best_params}')
print(f'最优 CV AUC（均值）：{grid_search.best_score_:.4f}')

cv_results = pd.DataFrame(grid_search.cv_results_)
cv_top = (cv_results[['param_max_depth', 'param_learning_rate',
                        'mean_test_score', 'std_test_score']]
          .sort_values('mean_test_score', ascending=False)
          .head(9)
          .rename(columns={'param_max_depth': 'max_depth',
                            'param_learning_rate': 'lr',
                            'mean_test_score': 'CV_AUC',
                            'std_test_score': 'CV_std'})
          .reset_index(drop=True))
print('\n各超参组合 CV AUC（按均值降序）：')
print(cv_top.round(4).to_string())


## 11.5 XGBoost 早停训练

早停（Early Stopping）在验证集误差连续 $k$ 轮不改善时自动停止，
避免手动指定树数并防止过拟合。


In [ ]:
# Cell 6：XGBoost 早停 + 最优超参
xgb_final = XGBClassifier(
    n_estimators=800,
    max_depth=best_params['max_depth'],
    learning_rate=best_params['learning_rate'],
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=2.0,
    eval_metric='logloss',
    early_stopping_rounds=30,
    random_state=SEED,
    verbosity=0,
)

xgb_final.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

best_iter     = xgb_final.best_iteration
xgb_train_auc = roc_auc_score(y_train, xgb_final.predict_proba(X_train)[:, 1])
xgb_test_auc  = roc_auc_score(y_test,  xgb_final.predict_proba(X_test)[:, 1])

print(f'最优树数（早停）：{best_iter}')
print(f'训练 AUC：{xgb_train_auc:.4f}')
print(f'测试 AUC：{xgb_test_auc:.4f}')
print(f'训练-测试差：{xgb_train_auc - xgb_test_auc:.4f}')

# logloss 迭代曲线
test_logloss = xgb_final.evals_result()['validation_0']['logloss']
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(test_logloss)+1), test_logloss,
        color='tomato', lw=1.5, label='测试集 logloss')
ax.axvline(best_iter, color='navy', linestyle='--', lw=1.5,
           label=f'最优轮次（{best_iter}）')
ax.set_xlabel('迭代轮次（树数）')
ax.set_ylabel('logloss')
ax.set_title('XGBoost 早停：测试集 logloss 迭代曲线')
ax.legend()
plt.tight_layout()
plt.show()


## 11.6 模型对比：XGBoost vs 逻辑回归

以 **AUC** 和 **混淆矩阵** 比较 XGBoost 与逻辑回归在测试集的表现。

逻辑回归需要特征标准化；XGBoost 不需要（分裂基于排序）。


In [ ]:
# Cell 7：XGBoost vs 逻辑回归 —— ROC + 混淆矩阵
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr = LogisticRegression(C=0.1, max_iter=500, random_state=SEED)
lr.fit(X_train_sc, y_train)
lr_train_auc = roc_auc_score(y_train, lr.predict_proba(X_train_sc)[:, 1])
lr_test_auc  = roc_auc_score(y_test,  lr.predict_proba(X_test_sc)[:, 1])

print('=== 测试集 AUC 汇总 ===')
print(f'XGBoost    训练={xgb_train_auc:.4f}  测试={xgb_test_auc:.4f}')
print(f'随机森林   训练={rf_train_auc:.4f}  测试={rf_test_auc:.4f}')
print(f'逻辑回归   训练={lr_train_auc:.4f}  测试={lr_test_auc:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左图：ROC
ax_roc = axes[0]
models_roc = [
    ('XGBoost', xgb_final.predict_proba(X_test)[:, 1]),
    ('随机森林', rf.predict_proba(X_test)[:, 1]),
    ('逻辑回归', lr.predict_proba(X_test_sc)[:, 1]),
]
for (name, prob), color in zip(models_roc, ['#1f77b4', '#2ca02c', '#ff7f0e']):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_v = roc_auc_score(y_test, prob)
    ax_roc.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc_v:.4f})')
ax_roc.plot([0, 1], [0, 1], 'k--', lw=1)
ax_roc.set_xlabel('假阳率 (FPR)')
ax_roc.set_ylabel('真阳率 (TPR)')
ax_roc.set_title('ROC 曲线对比（测试集）')
ax_roc.legend(fontsize=9)

# 右图：混淆矩阵
y_pred_xgb = xgb_final.predict(X_test)
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_xgb, display_labels=['跌(0)', '涨(1)'],
    ax=axes[1], colorbar=False
)
axes[1].set_title('XGBoost 混淆矩阵（测试集）')
plt.tight_layout()
plt.show()

print('\nXGBoost 分类报告（测试集）：')
print(classification_report(y_test, y_pred_xgb, target_names=['跌(0)', '涨(1)']))


## 11.7 XGBoost 特征重要性（Gain / Weight / Cover）

XGBoost 内置三种重要性：
- **gain**：特征贡献的平均分裂增益（推荐）
- **weight**：特征被用作分裂节点的次数
- **cover**：特征分裂覆盖的平均样本数


In [ ]:
# Cell 8：XGBoost 特征重要性（三种指标对比）
importance_types = ['gain', 'weight', 'cover']
imp_dfs = {}
booster = xgb_final.get_booster()

for itype in importance_types:
    scores = booster.get_score(importance_type=itype)
    imp_series = pd.Series(
        {f: scores.get(f'f{i}', 0.0) for i, f in enumerate(FEATURE_COLS)},
        name=itype
    )
    total = imp_series.sum() + 1e-9
    imp_dfs[itype] = imp_series / total

imp_combined = pd.DataFrame(imp_dfs).sort_values('gain', ascending=False)
print('=== XGBoost 特征重要性（归一化）===')
print(imp_combined.round(4).to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
palette = {'gain': '#1f77b4', 'weight': '#2ca02c', 'cover': '#ff7f0e'}
for ax, itype in zip(axes, importance_types):
    s = imp_combined[itype].sort_values()
    ax.barh(s.index, s.values, color=palette[itype], alpha=0.85)
    ax.set_title(f'特征重要性（{itype}）')
    ax.set_xlabel('重要性（归一化）')
    for v, name in zip(s.values, s.index):
        ax.text(v + 0.003, name, f'{v:.3f}', va='center', fontsize=9)
plt.suptitle('XGBoost 三种特征重要性对比', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## 11.8 置换重要性（SHAP 替代方案）

本环境未安装 `shap` 库，使用 `sklearn.inspection.permutation_importance` 替代。

**置换重要性**：对测试集中某特征的值随机打乱，观察 AUC 下降幅度。
不受特征基数影响，反映特征对泛化的真实贡献。

---

**SHAP 值思想**：若已安装 `shap`，可使用 `shap.TreeExplainer` 对每个预测样本
将每个特征贡献精确分解（满足效率性：所有 SHAP 值之和 = 预测值 - 基准值）。


In [ ]:
# Cell 9：置换重要性（try shap → fallback permutation_importance）
try:
    import shap
    HAS_SHAP = True
    print('检测到 shap，使用 TreeExplainer')
except ImportError:
    HAS_SHAP = False
    print('shap 未安装，使用 permutation_importance 替代')

if HAS_SHAP:
    explainer = shap.TreeExplainer(xgb_final)
    shap_values = explainer.shap_values(X_test)
    shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS, show=False)
    plt.title('SHAP Summary Plot（测试集）')
    plt.tight_layout()
    plt.show()
else:
    perm_result = permutation_importance(
        xgb_final, X_test, y_test,
        n_repeats=20, random_state=SEED,
        scoring='roc_auc', n_jobs=1
    )
    perm_df = pd.DataFrame({
        '特征': FEATURE_COLS,
        'AUC下降均值': perm_result.importances_mean,
        'AUC下降标准差': perm_result.importances_std,
    }).sort_values('AUC下降均值', ascending=False)

    print('=== 置换重要性（测试集，n_repeats=20）===')
    print(perm_df.round(5).to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 5))
    perm_s = perm_df.sort_values('AUC下降均值')
    ax.barh(perm_s['特征'], perm_s['AUC下降均值'],
            xerr=perm_s['AUC下降标准差'],
            color='#4c72b0', alpha=0.85, capsize=4)
    ax.axvline(0, color='black', lw=0.8, linestyle='--')
    ax.set_xlabel('AUC 下降幅度（值越大 → 该特征越重要）')
    ax.set_title('置换重要性（XGBoost，测试集）')
    plt.tight_layout()
    plt.show()

    print()
    print('【SHAP 思想说明】')
    print('SHAP = Shapley Additive Explanations')
    print('将博弈论 Shapley 值引入模型解释：')
    print('  phi_j = 特征 j 对该预测的边际贡献（对所有子集取加权平均）')
    print('  效率性：sum(phi_j) = 预测值 - 基准值（训练集平均预测）')
    print('TreeSHAP 对树模型精确且高效（多项式复杂度），是当前最推荐的解释工具')


## 11.9 偏差-方差可视化（Bootstrap）

通过 Bootstrap 重采样，直观展示深树（高方差）vs 随机森林（低方差）的预测稳定性。


In [ ]:
# Cell 10：偏差-方差可视化
rng = np.random.default_rng(SEED)
B   = 30
n_show = 5

pred_deep = np.zeros((B, len(X_test)))
pred_rf_s = np.zeros((B, len(X_test)))

for b in range(B):
    idx = rng.integers(0, n_train, size=n_train)
    Xb, yb = X_train[idx], y_train[idx]
    dt_b = DecisionTreeClassifier(max_depth=10, random_state=b)
    dt_b.fit(Xb, yb)
    pred_deep[b] = dt_b.predict_proba(X_test)[:, 1]
    rf_b = RandomForestClassifier(n_estimators=50, max_depth=5,
                                   random_state=b, n_jobs=1)
    rf_b.fit(Xb, yb)
    pred_rf_s[b] = rf_b.predict_proba(X_test)[:, 1]

x_tick = np.arange(n_show)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, preds, title in [
    (axes[0], pred_deep, '深树（depth=10）—— 高方差'),
    (axes[1], pred_rf_s, '随机森林（50棵）—— 低方差')
]:
    means = preds[:, :n_show].mean(axis=0)
    stds  = preds[:, :n_show].std(axis=0)
    ax.bar(x_tick, means, width=0.5, color='steelblue', alpha=0.7)
    ax.errorbar(x_tick, means, yerr=stds * 2, fmt='none',
                ecolor='tomato', capsize=6, elinewidth=2,
                label=f'±2σ (σ均={stds.mean():.4f})')
    ax.set_xticks(x_tick)
    ax.set_xticklabels([f'样本{i+1}' for i in range(n_show)])
    ax.set_ylabel('预测概率 P(涨)')
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.axhline(0.5, color='gray', lw=0.8, linestyle=':')
plt.suptitle(f'Bootstrap {B} 次：预测方差对比（前{n_show}个样本）', y=1.02)
plt.tight_layout()
plt.show()

print(f'深树预测方差（均值）：   {pred_deep.std(axis=0).mean():.4f}')
print(f'随机森林预测方差（均值）：{pred_rf_s.std(axis=0).mean():.4f}')
print('→ 随机森林方差显著更小，印证 Bagging 降低方差的效果')


## 11.10 综合结果汇总


In [ ]:
# Cell 11：综合汇总表
dt3 = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_train, y_train)
dt10 = DecisionTreeClassifier(max_depth=10, random_state=SEED).fit(X_train, y_train)

summary = pd.DataFrame([
    {'模型': '决策树(depth=3)',      '测试AUC': roc_auc_score(y_test, dt3.predict_proba(X_test)[:,1]),  '特点': '低复杂度，高偏差'},
    {'模型': '决策树(depth=10)',     '测试AUC': roc_auc_score(y_test, dt10.predict_proba(X_test)[:,1]), '特点': '高方差，易过拟合'},
    {'模型': '随机森林(300棵)',       '测试AUC': rf_test_auc,    '特点': 'Bagging 降方差'},
    {'模型': 'XGBoost(早停调参)',    '测试AUC': xgb_test_auc,   '特点': 'Boosting 降偏差，正则化'},
    {'模型': '逻辑回归',              '测试AUC': lr_test_auc,    '特点': '线性基准，可解释'},
])
summary['测试AUC'] = summary['测试AUC'].round(4)
summary = summary.sort_values('测试AUC', ascending=False).reset_index(drop=True)
print('=== 模型对比汇总 ===')
print(summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(summary['模型'][::-1], summary['测试AUC'][::-1], color='#4c72b0', alpha=0.85)
ax.axvline(0.5, color='gray', lw=1.2, linestyle='--', label='随机猜测（0.5）')
ax.set_xlabel('测试集 AUC')
ax.set_title('各模型测试集 AUC 对比（A 股 TECH 涨跌预测）')
for bar, val in zip(ax.patches, summary['测试AUC'][::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
ax.legend()
ax.set_xlim(0.47, summary['测试AUC'].max() + 0.025)
plt.tight_layout()
plt.show()

print()
print('【结论】')
print('1. 金融收益序列信噪比低，AUC 0.52~0.58 属正常范围')
print('2. 深树过拟合严重；随机森林和 XGBoost 通过集成显著改善')
print('3. 高 AUC 不等于高实盘收益，需额外考虑交易成本')


---
## 习题参考解答

以下各 cell 对应正文习题，可直接运行验证。


In [ ]:
# === 习题 11.1：基尼系数与信息熵（代数验证）===
print('习题 11.1：二分类不纯度（p1=p2=0.5）')
p1 = p2 = 0.5
gini    = 1 - (p1**2 + p2**2)
entropy = -(p1 * np.log2(p1) + p2 * np.log2(p2))
print(f'  基尼系数 G = 1 - (0.5^2+0.5^2) = {gini}')
print(f'  信息熵  H = -2*0.5*log2(0.5)  = {entropy}')

p_vals = np.linspace(0.001, 0.999, 200)
gini_v = 2 * p_vals * (1 - p_vals)
entr_v = -(p_vals * np.log2(p_vals) + (1-p_vals) * np.log2(1-p_vals))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p_vals, gini_v, label='基尼系数 G', color='#1f77b4', lw=2)
ax.plot(p_vals, entr_v, label='信息熵 H（log2）', color='#ff7f0e', lw=2)
ax.axvline(0.5, color='gray', linestyle='--', lw=1)
ax.set_xlabel('类别 1 的比例 $p_1$')
ax.set_ylabel('不纯度')
ax.set_title('二分类不纯度：基尼 vs 信息熵')
ax.legend()
plt.tight_layout()
plt.show()
print('两者均在 p=0.5 时最大，p=0 或 1 时为 0（纯节点），形状相似但量纲不同')


In [ ]:
# === 习题 11.2：OOB 误差随树数变化 ===
print('习题 11.2：随机森林 OOB 误差随树数变化（信用数据）')
from fds import load_credit
credit = load_credit()
feat_c  = [c for c in credit.columns if c != 'default']
Xc, yc  = credit[feat_c].values, credit['default'].values

n_trees_list = [10, 30, 50, 100, 200, 300, 500]
oob_errors   = []
for n in n_trees_list:
    rf_c = RandomForestClassifier(n_estimators=n, max_depth=5,
                                   oob_score=True, random_state=SEED, n_jobs=1)
    rf_c.fit(Xc, yc)
    oob_errors.append(1 - rf_c.oob_score_)
    print(f'  n={n:4d}: OOB Error = {oob_errors[-1]:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_trees_list, oob_errors, 'o-', color='#2ca02c', lw=2)
best_n = n_trees_list[int(np.argmin(oob_errors))]
ax.axvline(best_n, color='tomato', linestyle='--',
           label=f'OOB 最低时树数：{best_n}')
ax.set_xlabel('树的数量 (n_estimators)')
ax.set_ylabel('OOB 误分类率')
ax.set_title('随机森林 OOB 误差随树数变化（信用违约数据）')
ax.legend()
plt.tight_layout()
plt.show()
print(f'\n结论：OOB 误差在约 {best_n} 棵树后趋于稳定，继续增树收益递减但不会增大')


In [ ]:
# === 习题 11.3：偏差-方差诊断 ===
print('习题 11.3：偏差-方差诊断（训练 vs 测试 AUC）')
dt3_tr_auc  = roc_auc_score(y_train, DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_train, y_train).predict_proba(X_train)[:,1])
dt3_te_auc  = roc_auc_score(y_test,  DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_train, y_train).predict_proba(X_test)[:,1])
dt10_tr_auc = roc_auc_score(y_train, DecisionTreeClassifier(max_depth=10, random_state=SEED).fit(X_train, y_train).predict_proba(X_train)[:,1])
dt10_te_auc = roc_auc_score(y_test,  DecisionTreeClassifier(max_depth=10, random_state=SEED).fit(X_train, y_train).predict_proba(X_test)[:,1])

cases = [
    ('决策树(depth=3)',  dt3_tr_auc,  dt3_te_auc),
    ('决策树(depth=10)', dt10_tr_auc, dt10_te_auc),
    ('XGBoost(早停)',    xgb_train_auc, xgb_test_auc),
    ('逻辑回归',         lr_train_auc,  lr_test_auc),
]
print(f'{"模型":<20} {"训练AUC":>8} {"测试AUC":>8} {"差距":>6} {"诊断"}')
for name, tr, te in cases:
    gap = tr - te
    diag = '高方差（过拟合）' if gap > 0.05 else ('高偏差（欠拟合）' if te < 0.53 else '相对均衡')
    print(f'{name:<20} {tr:>8.4f} {te:>8.4f} {gap:>6.4f} {diag}')
print()
print('调整建议：')
print('  高方差 → 减小 max_depth / 增大 reg_lambda / 开早停 / 增加数据')
print('  高偏差 → 增加树数 / 更多特征 / 提高模型容量')


In [ ]:
# === 习题 11.4：多支股票 Pooled 训练 vs 单股 ===
print('习题 11.4：4支股票 Pooled vs 单支 TECH')

all_feat_list = []
for ticker in ['BANK', 'LIQUOR', 'TECH', 'UTILITY']:
    df_t = build_features(rets[ticker], prices[ticker]).copy()
    df_t['ticker'] = ticker
    all_feat_list.append(df_t)

all_feat    = pd.concat(all_feat_list).sort_index()
n_pooled    = len(all_feat)
n_tr_pooled = int(n_pooled * 0.70)
X_pool = all_feat[FEATURE_COLS].values
y_pool = all_feat['target'].values
X_ptr, X_pte = X_pool[:n_tr_pooled], X_pool[n_tr_pooled:]
y_ptr, y_pte = y_pool[:n_tr_pooled], y_pool[n_tr_pooled:]

xgb_pool = XGBClassifier(
    n_estimators=200,
    max_depth=best_params['max_depth'],
    learning_rate=best_params['learning_rate'],
    subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
    eval_metric='logloss', random_state=SEED, verbosity=0
)
xgb_pool.fit(X_ptr, y_ptr)
pool_auc = roc_auc_score(y_pte, xgb_pool.predict_proba(X_pte)[:, 1])

print(f'单支 TECH 测试 AUC：{xgb_test_auc:.4f}')
print(f'4支股票 Pooled 测试 AUC：{pool_auc:.4f}')
print()
print('注意事项：')
print('1. 同一时刻多支股票标签高度相关（市场整体涨跌），独立性假设被违反')
print('2. 需做截面标准化，避免个股收益均值漂移主导模型')
print('3. 测试集须仍为时间靠后的数据，不可按股票随机切分')
print('4. Pooled 可增加样本量，但引入跨股污染，建议与单股交叉验证')


In [ ]:
# === 习题 11.5：SHAP 效率性公理示意 ===
print('习题 11.5：SHAP 效率性公理 —— 金融应用场景')
print()
print('核心公理：所有特征 SHAP 值之和 = 预测值 - 基准值')
print()

# 数值示意
baseline = 0.12
shap_demo = {
    '年龄特征':    +0.02,
    '收入特征':    -0.08,
    '负债率特征':  +0.15,
    '信用历史':    +0.02,
    '逾期次数':    +0.12,
}
total_shap = sum(shap_demo.values())
prediction = baseline + total_shap

print(f'场景：信贷风险（基准={baseline}，预测违约概率={prediction}）')
for feat, val in shap_demo.items():
    arrow = '↑' if val > 0 else '↓'
    print(f'  {feat:12s}: SHAP={val:+.2f}  {arrow}')
print(f'  合计 SHAP：{total_shap:+.2f}')
print(f'  验证效率性：{baseline} + {total_shap:.2f} = {prediction:.2f}  ✓')
print()

# 可视化瀑布图（Waterfall 示意）
fig, ax = plt.subplots(figsize=(9, 4))
feats = list(shap_demo.keys())
vals  = list(shap_demo.values())
cumulative = np.cumsum([baseline] + vals)
for i, (f, v) in enumerate(zip(feats, vals)):
    color = '#d62728' if v > 0 else '#2ca02c'
    ax.barh(f, v, left=cumulative[i], color=color, alpha=0.85, height=0.5)
    ax.text(cumulative[i] + v/2, i, f'{v:+.2f}', ha='center', va='center',
            fontsize=9, color='white', fontweight='bold')
ax.axvline(baseline,    color='gray',  linestyle='--', lw=1, label=f'基准值={baseline}')
ax.axvline(prediction,  color='black', linestyle='-',  lw=1.5, label=f'预测值={prediction}')
ax.set_xlabel('违约概率')
ax.set_title('SHAP Waterfall 示意（信贷风险）')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('量化选股监管合规：SHAP 可逐样本逐特征解释投资决策，符合资管新规精神')
